In [9]:
import tensorflow as tf

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

In [11]:
train_data = train_datagen.flow_from_directory(
    'PetImages/', 
    target_size=(224,224),
     batch_size=32,
    class_mode='binary', 
    subset='training'
)

val_data = val_datagen.flow_from_directory(
    'PetImages/', 
    target_size=(224,224), 
    batch_size=32,
    class_mode='binary',
     subset='validation'
)

Found 20000 images belonging to 2 classes.
Found 4998 images belonging to 2 classes.


In [12]:
#load  pretrained model 
from tensorflow.keras.applications import VGG16
base_model = VGG16(
    weights = 'imagenet',
    include_top = False,
    input_shape=(224,224,3)

)

In [13]:
#freeze layer 

for layers in base_model.layers:
    layers.trainable = False 
    

In [14]:
#build model 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout

model = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)





In [15]:
history = model.fit(
    train_data,
    validation_data=val_data,   
    epochs=1
)      

625/625 ━━━━━━━━━━━━━━━━━━━━ 1976s 3s/step - accuracy: 0.8207 - loss: 0.4225 - val_accuracy: 0.9148 - val_loss: 0.2023


In [16]:
#fine_tuning 

base_model.trainable  = True 

print(f"Total layers in base_model: {len(base_model.layers)}")

for layer in base_model.layers[:-4]:
    layer.trainable= False 
for layer in base_model.layers[-4:]:
    layer.trainable=True   

Total layers in base_model: 19


In [17]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
fine_tune_history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=1,

)


625/625 ━━━━━━━━━━━━━━━━━━━━ 2257s 4s/step - accuracy: 0.4983 - loss: 0.8725 - val_accuracy: 0.5000 - val_loss: 0.6932
